# Seven-vectoriser test-case representation comparison for Firebase Chat

This notebook compares the seven text vectorisers evaluated by Chakraborty, Elhence, and Arora (2019) on the same 69 Firebase Chat test cases: TF-IDF, Feature Hashing, Word2Vec, GloVe, FastText, ELMo, and Flair.

The bug labels are fixed dummy values. They are not application test results or evidence of real defects. This is a representation-comparison pilot, not a final Bayesian Optimization experiment.


## Experiment flow

The notebook reads the QA workbook and checks all 69 cases. It first shows the nine known warm-start outcomes, one case per feature. Each representation then converts the same cases into its own numeric matrix. The experiment fits a Gaussian Process from the known outcomes, scores the untested candidates, selects one case, and reveals only that selected case's dummy result.

This process continues until every candidate has a position in the final order. The notebook compares only the seven paper vectorisers under the same warm start, dummy outcomes, cost-aware UCB rule, and stopping point.


In [1]:
import os

PAPER_VECTORISERS = (
    "TF-IDF", "Feature Hashing", "Word2Vec", "GloVe",
    "FastText", "ELMo", "Flair",
)
QUICK_VECTORISERS = ("TF-IDF", "Feature Hashing")


def select_vectorisers(selection: str) -> tuple[str, ...]:
    """Return a deliberate subset; quick never downloads pretrained weights."""
    requested = [item.strip() for item in selection.split(",") if item.strip()]
    if not requested or requested == ["quick"]:
        return QUICK_VECTORISERS
    if requested == ["all"]:
        return PAPER_VECTORISERS
    unknown = sorted(set(requested).difference(PAPER_VECTORISERS))
    if unknown:
        raise ValueError(f"Unknown vectoriser selection: {', '.join(unknown)}")
    return tuple(name for name in PAPER_VECTORISERS if name in requested)


def build_vectoriser_metadata(selection: str) -> dict[str, object]:
    """Record the exact subset that produced a result folder."""
    return {
        "vectoriser_selection": selection,
        "selected_vectorisers": list(select_vectorisers(selection)),
    }


def default_vectoriser_selection(is_colab: bool) -> str:
    """Use all methods in Colab and the safe pair on a local laptop."""
    return "all" if is_colab else "quick"


def is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


IS_COLAB = is_colab_runtime()
VECTORISER_SELECTION = os.getenv(
    "PAPER_VECTORISER_MODE", default_vectoriser_selection(IS_COLAB)
)
SELECTED_VECTORISERS = select_vectorisers(VECTORISER_SELECTION)
print("Selected vectorisers:", ", ".join(SELECTED_VECTORISERS))
if IS_COLAB and VECTORISER_SELECTION == "all":
    print("Google Colab automatically runs all seven vectorisers.")
elif not IS_COLAB and VECTORISER_SELECTION == "quick":
    print("Local laptop mode uses the lightweight pair by default.")


Selected vectorisers: TF-IDF, Feature Hashing, Word2Vec, GloVe, FastText, ELMo, Flair
Use PAPER_VECTORISER_MODE=all only when you deliberately want every pretrained-model download.


In [2]:
# Install only packages required by the selected vectorisers.
import importlib.util
import os
import subprocess
import sys

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
}
if set(SELECTED_VECTORISERS).intersection({"Word2Vec", "GloVe", "FastText"}):
    required["gensim"] = "gensim"
if set(SELECTED_VECTORISERS).intersection({"ELMo", "Flair"}):
    required["flair"] = "flair"
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing required packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

print("Environment ready for:", ", ".join(SELECTED_VECTORISERS))
print("Pretrained weights download only when a selected pretrained vectoriser is built.")

Environment ready for: TF-IDF, Feature Hashing, Word2Vec, GloVe, FastText, ELMo, Flair
Pretrained weights download only when a selected pretrained vectoriser is built.


In [3]:
from pathlib import Path
import hashlib
import json
import math
import re
import shutil
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import FileLink, display

RANDOM_STATE = 42
KAPPA = 1.5
COST_EXPONENT = 0.5
PAPER_CITATION = "Chakraborty, Elhence, and Arora (2019), Sparse Victory"
AUTO_DOWNLOAD_IN_COLAB = False
DUMMY_FAULT_THEMES = {
    "message_delivery_and_read_state": [
        "FC-MN-006", "FC-MN-007", "FC-CRP-007", "FC-CRP-008",
        "FC-CRG-004", "FC-CRG-007", "FC-MIF-001", "FC-MIF-002", "FC-MIF-003",
    ],
    "group_membership_permission": [
        "FC-NCP-004", "FC-NCP-005", "FC-GPR-002", "FC-GPR-003",
    ],
}
DUMMY_BUG_IDS = sorted({tcs_id for ids_in_theme in DUMMY_FAULT_THEMES.values() for tcs_id in ids_in_theme})

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(RANDOM_STATE)
print(f"Controlled dummy oracle: {len(DUMMY_BUG_IDS)} dummy bugs in {len(DUMMY_FAULT_THEMES)} fault themes")
print(f"Cost-aware UCB: kappa={KAPPA}, cost exponent={COST_EXPONENT}")

Controlled dummy oracle: 13 dummy bugs in 2 fault themes
Cost-aware UCB: kappa=1.5, cost exponent=0.5


## 1. Load and validate the QA workbook

On a local laptop, the notebook searches the current folder and its parents for scenarios/firebase_chat/scenario.xlsx. In Colab, it opens a file upload dialog when the workbook is not available.

In [4]:
import re
from pathlib import Path
import numpy as np
import pandas as pd


def parse_minutes(value) -> float:
    """Extract the first numeric duration from the QA Time Testing cell."""
    match = re.search(r"\d+(?:\.\d+)?", str(value))
    if not match:
        raise ValueError(f"Cannot parse Time Testing value: {value!r}")
    return float(match.group())


def choose_initial_seed(cases: pd.DataFrame) -> list[int]:
    """Choose the lexicographically first TCS ID from every Menu."""
    ordered = cases.reset_index().sort_values(["Menu", "TCS ID"], kind="stable")
    return sorted(ordered.groupby("Menu", sort=True).head(1)["index"].tolist())


def find_workbook() -> Path:
    checked = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "scenarios" / "firebase_chat" / "scenario.xlsx"
        checked.append(candidate)
        if candidate.exists():
            return candidate.resolve()

    try:
        from google.colab import files
    except ImportError as exc:
        locations = "\n".join(str(path) for path in checked)
        raise FileNotFoundError(
            "scenario.xlsx was not found. Run from the MAS AI repository or copy the workbook "
            f"to scenarios/firebase_chat/scenario.xlsx. Checked:\n{locations}"
        ) from exc

    print("Upload scenario.xlsx from scenarios/firebase_chat/")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No workbook was uploaded.")
    uploaded_name = next(iter(uploaded))
    return Path(uploaded_name).resolve()


def load_qa_cases(workbook: Path) -> pd.DataFrame:
    raw = pd.read_excel(workbook, header=None)
    matches = np.argwhere(raw.eq("TCS ID").to_numpy())
    if len(matches) == 0:
        raise ValueError("Could not locate the TCS ID header in the workbook.")
    header_row = int(matches[0, 0])
    cases = pd.read_excel(workbook, header=header_row)
    required_columns = [
        "TCS ID", "Menu", "Submenu 1", "Submenu 2", "Test Case Scenario",
        "Test Step", "Expected Result", "Test Type", "User", "Time Testing",
    ]
    missing_columns = [column for column in required_columns if column not in cases.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")
    cases = cases.loc[cases["TCS ID"].astype(str).str.startswith("FC-")].copy()
    cases = cases[required_columns].reset_index(drop=True)
    text_columns = [column for column in required_columns if column != "Time Testing"]
    cases[text_columns] = cases[text_columns].fillna("").astype(str)
    cases["estimated_cost"] = cases["Time Testing"].map(parse_minutes)
    if cases["TCS ID"].duplicated().any():
        duplicates = cases.loc[cases["TCS ID"].duplicated(), "TCS ID"].tolist()
        raise ValueError(f"Duplicate TCS IDs: {duplicates}")
    return cases


In [5]:
WORKBOOK_PATH = find_workbook()
cases = load_qa_cases(WORKBOOK_PATH)
if len(cases) != 69:
    raise ValueError(f"This experiment expects 69 Firebase Chat cases, found {len(cases)}.")

unknown_bug_ids = sorted(set(DUMMY_BUG_IDS) - set(cases["TCS ID"]))
if unknown_bug_ids:
    raise ValueError(f"DUMMY_BUG_IDS not found in workbook: {unknown_bug_ids}")

oracle = cases["TCS ID"].isin(DUMMY_BUG_IDS).astype(int).to_numpy()
fault_theme_by_id = {
    tcs_id: theme
    for theme, ids_in_theme in DUMMY_FAULT_THEMES.items()
    for tcs_id in ids_in_theme
}
costs = cases["estimated_cost"].to_numpy(dtype=float)
initial_indices = choose_initial_seed(cases)
warm_start_bug_count = int(oracle[initial_indices].sum())
assert warm_start_bug_count >= 1
ids = cases["TCS ID"].to_numpy()
menus = cases["Menu"].to_numpy()

repo_root = WORKBOOK_PATH.parents[2] if WORKBOOK_PATH.parent.name == "firebase_chat" else Path.cwd()
if (repo_root / "scenarios" / "firebase_chat" / "scenario.xlsx").exists():
    RESULT_DIR = repo_root / "experiment" / "bayesian" / "results"
else:
    RESULT_DIR = Path.cwd() / "experiment" / "bayesian" / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Workbook:", WORKBOOK_PATH)
print("Cases:", len(cases), "| Menus/features:", cases["Menu"].nunique())
print("Warm start:", len(initial_indices), "known historical outcomes, one per Menu")
print("Warm-start dummy bugs:", warm_start_bug_count, "| warm-start passes:", len(initial_indices) - warm_start_bug_count)
print("Total estimated cost:", costs.sum(), "minutes")
fault_theme_table = cases.loc[cases["TCS ID"].isin(DUMMY_BUG_IDS), ["TCS ID", "Menu", "Test Case Scenario"]].copy()
fault_theme_table["dummy_fault_theme"] = fault_theme_table["TCS ID"].map(fault_theme_by_id)
display(fault_theme_table.sort_values(["dummy_fault_theme", "TCS ID"]).reset_index(drop=True))

Workbook: C:\Users\radit\Project\VisualStudioProject\Skripsi\MAS AI\scenarios\firebase_chat\scenario.xlsx
Cases: 69 | Menus/features: 9
Warm start: 9 known historical outcomes, one per Menu
Warm-start dummy bugs: 1 | warm-start passes: 8
Total estimated cost: 632.0 minutes


,TCS ID,Menu,Test Case Scenario,dummy_fault_theme
0,FC-GPR-002,Profil Grup,"Tombol ""Add Participant"" tampil bagi pembuat grup",group_membership_permission
1,FC-GPR-003,Profil Grup,"Tombol ""Add Participant"" tidak tampil bagi ang...",group_membership_permission
2,FC-NCP-004,Buat Chat Baru,Menambahkan partisipan baru ke grup yang sudah...,group_membership_permission
3,FC-NCP-005,Buat Chat Baru,"Daftar pengguna pada mode ""Add Participant"" ti...",group_membership_permission
4,FC-CRG-004,Ruang Obrolan,"Status pesan menjadi ""read"" hanya setelah selu...",message_delivery_and_read_state
5,FC-CRG-007,Ruang Obrolan,Jumlah pesan belum dibaca (unread) direset saa...,message_delivery_and_read_state
6,FC-CRP-007,Ruang Obrolan,Pesan yang diterima otomatis ditandai sudah di...,message_delivery_and_read_state
7,FC-CRP-008,Ruang Obrolan,Melihat info pengiriman (Delivered/Read) pada ...,message_delivery_and_read_state
8,FC-MIF-001,Info Pesan,Menampilkan waktu Delivered dan Read pada perc...,message_delivery_and_read_state
9,FC-MIF-002,Info Pesan,"Status ""Delivered"" terisi sebelum status ""Read...",message_delivery_and_read_state


## 2. Warm-start results before sequential selection

The sequential selection scaffold cannot make its first informed choice without any previous result. The experiment therefore begins with nine already known outcomes, one QA case from each Menu. These are the only labels known before the model chooses test #10.


In [6]:
warm_start_table = cases.iloc[initial_indices][[
    "TCS ID", "Menu", "Test Case Scenario", "Test Step", "Expected Result", "estimated_cost",
]].copy()
warm_start_table["known_outcome"] = np.where(
    oracle[initial_indices].astype(bool), "Dummy bug", "Pass"
)
warm_start_table["dummy_fault_theme"] = warm_start_table["TCS ID"].map(fault_theme_by_id).fillna("-")
warm_start_table = warm_start_table.sort_values(["Menu", "TCS ID"]).reset_index(drop=True)
display(warm_start_table)


,TCS ID,Menu,Test Case Scenario,Test Step,Expected Result,estimated_cost,known_outcome,dummy_fault_theme
0,FC-NCG-001,Buat Chat Baru,Membuat grup baru dengan nama grup dan foto valid,1. Pilih minimal 1 partisipan pada halaman Pil...,1. Grup baru berhasil dibuat dan langsung tamp...,11.0,Pass,-
1,FC-MIF-001,Info Pesan,Menampilkan waktu Delivered dan Read pada perc...,1. Kirim pesan pada ruang obrolan personal\n2....,1. Menampilkan waktu Delivered dan waktu Read ...,5.0,Dummy bug,message_delivery_and_read_state
2,FC-FSI-001,Lihat Gambar Layar Penuh,Menampilkan gambar resolusi penuh beserta capt...,1. Buka ruang obrolan yang memiliki pesan berg...,1. Gambar tampil dalam resolusi penuh satu lay...,6.0,Pass,-
3,FC-LGN-001,Login,Login berhasil dengan email dan password yang ...,1. Buka aplikasi Firebase Chat\n2. Masukkan em...,"1. Menampilkan Toast ""Login Success""\n2. User ...",13.0,Pass,-
4,FC-MN-001,Main,Menampilkan daftar percakapan diurutkan dari p...,1. Login ke dalam aplikasi dengan akun yang me...,1. Percakapan dengan pesan terakhir paling bar...,9.0,Pass,-
5,FC-GPR-001,Profil Grup,"Menampilkan informasi grup (avatar, nama, juml...",1. Buka Profil Grup dari ruang obrolan grup,"1. Menampilkan avatar grup, nama grup, ""Partic...",7.0,Pass,-
6,FC-PRF-001,Profil Pengguna,Menampilkan profil pengguna lain (lawan bicara),1. Buka ruang obrolan personal\n2. Tekan avata...,"1. Menampilkan avatar, nama, dan email milik l...",6.0,Pass,-
7,FC-REG-001,Register,Registrasi akun baru berhasil dengan data valid,"1. Buka halaman Register\n2. Isi Nama, Email, ...",1. Akun baru berhasil dibuat pada Firebase Aut...,5.0,Pass,-
8,FC-CRG-001,Ruang Obrolan,Mengirim pesan teks dalam ruang obrolan grup,"1. Buka ruang obrolan grup\n2. Ketik pesan, te...",1. Pesan tampil pada ruang obrolan lengkap den...,6.0,Pass,-


## 3. All fixed dummy bugs for researcher inspection

The table below explains where the 13 fixed dummy bugs are placed and what each case checks. It is visible only for understanding the controlled experiment. The surrogate model never receives this full table as training data.


In [7]:
dummy_bug_table = cases.loc[cases["TCS ID"].isin(DUMMY_BUG_IDS), [
    "TCS ID", "Menu", "Test Case Scenario", "Test Step", "Expected Result", "estimated_cost",
]].copy()
dummy_bug_table["dummy_fault_theme"] = dummy_bug_table["TCS ID"].map(fault_theme_by_id)
dummy_bug_table = dummy_bug_table.sort_values(["dummy_fault_theme", "TCS ID"]).reset_index(drop=True)
display(dummy_bug_table)


,TCS ID,Menu,Test Case Scenario,Test Step,Expected Result,estimated_cost,dummy_fault_theme
0,FC-GPR-002,Profil Grup,"Tombol ""Add Participant"" tampil bagi pembuat grup",1. Buka Profil Grup sebagai user yang membuat ...,"1. Tombol ""Add Participant"" tampil dan dapat d...",2.0,group_membership_permission
1,FC-GPR-003,Profil Grup,"Tombol ""Add Participant"" tidak tampil bagi ang...",1. Buka Profil Grup sebagai anggota yang bukan...,"1. Tombol ""Add Participant"" tidak ditampilkan",7.0,group_membership_permission
2,FC-NCP-004,Buat Chat Baru,Menambahkan partisipan baru ke grup yang sudah...,1. Buka Profil Grup sebagai pembuat grup\n2. T...,"1. Menampilkan Toast ""Success Adding new Parti...",2.0,group_membership_permission
3,FC-NCP-005,Buat Chat Baru,"Daftar pengguna pada mode ""Add Participant"" ti...",1. Buka Profil Grup sebagai pembuat grup\n2. T...,1. User yang sudah menjadi anggota grup tidak ...,11.0,group_membership_permission
4,FC-CRG-004,Ruang Obrolan,"Status pesan menjadi ""read"" hanya setelah selu...",1. Kirim pesan pada grup beranggotakan lebih d...,"1. Status pesan tetap centang tunggal (belum ""...",8.0,message_delivery_and_read_state
5,FC-CRG-007,Ruang Obrolan,Jumlah pesan belum dibaca (unread) direset saa...,1. Buka ruang obrolan grup yang memiliki pesan...,1. Badge unread untuk grup tersebut hilang/men...,8.0,message_delivery_and_read_state
6,FC-CRP-007,Ruang Obrolan,Pesan yang diterima otomatis ditandai sudah di...,1. User B mengirim pesan ke User A saat User A...,1. Seluruh pesan yang termuat langsung berstat...,13.0,message_delivery_and_read_state
7,FC-CRP-008,Ruang Obrolan,Melihat info pengiriman (Delivered/Read) pada ...,1. Kirim pesan pada ruang obrolan personal\n2....,1. Menampilkan halaman Info Pesan dengan waktu...,13.0,message_delivery_and_read_state
8,FC-MIF-001,Info Pesan,Menampilkan waktu Delivered dan Read pada perc...,1. Kirim pesan pada ruang obrolan personal\n2....,1. Menampilkan waktu Delivered dan waktu Read ...,5.0,message_delivery_and_read_state
9,FC-MIF-002,Info Pesan,"Status ""Delivered"" terisi sebelum status ""Read...","1. Kirim pesan pada ruang obrolan personal, tu...",1. Waktu Delivered terisi terlebih dahulu sesa...,10.0,message_delivery_and_read_state


## 4. Fixed dummy outcomes

The experiment uses one fixed synthetic oracle with two fault themes: message delivery and read state, plus group membership permission. Every method faces exactly the same oracle. The full mapping is visible to the researcher, but the surrogate model does not receive it.

The warm start contains one test from each Menu and represents known historical outcomes. It contains at least one dummy bug without selecting the warm-start cases from oracle labels. After that point, the surrogate receives outcomes only after it selects a test. The fixed Excel order is used later only to place discrete candidates on a readable horizontal axis.

In [8]:
x_display = np.arange(len(cases))
print(f"The fixed oracle contains {int(oracle.sum())} dummy bugs across {len(oracle)} test cases.")

The fixed oracle contains 13 dummy bugs across 69 test cases.


## 5. Independent vectoriser matrices

Each method converts the workbook into its own matrix. The matrices are never combined. The seven methods are the exact vectoriser set from Chakraborty, Elhence, and Arora (2019):

- TF-IDF and Feature Hashing are sparse lexical vectorisers.
- Word2Vec, GloVe, and FastText are pretrained static word embeddings averaged into a document vector.
- ELMo and Flair are contextual word embeddings averaged into a document vector.

The first full execution downloads pretrained weights for the five neural vectorisers if they are not in the local cache. No paid API is used.

In [ ]:
TEXT_FIELDS = [
    "Menu", "Submenu 1", "Submenu 2", "Test Case Scenario", "Test Step",
    "Expected Result", "Test Type", "User",
]


def serialize_cases(frame: pd.DataFrame) -> list[str]:
    return [
        "\n".join(f"{field}: {row[field]}" for field in TEXT_FIELDS)
        for _, row in frame.iterrows()
    ]


def normalize_rows(matrix: np.ndarray) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=float)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.where(norms == 0, 1.0, norms)


def tokenize_with_bigrams(text: str) -> list[str]:
    tokens = re.findall(r"(?u)\b\w\w+\b", text.lower())
    return tokens + [f"{left}__{right}" for left, right in zip(tokens, tokens[1:])]


def build_tfidf(frame: pd.DataFrame) -> np.ndarray:
    from sklearn.feature_extraction.text import TfidfVectorizer

    vectoriser = TfidfVectorizer(ngram_range=(1, 2), max_features=300, sublinear_tf=True)
    return normalize_rows(vectoriser.fit_transform(serialize_cases(frame)).toarray())


def build_feature_hashing(frame: pd.DataFrame, dimensions: int = 300) -> np.ndarray:
    matrix = np.zeros((len(frame), dimensions), dtype=float)
    for row, document in enumerate(serialize_cases(frame)):
        for token in tokenize_with_bigrams(document):
            digest = hashlib.blake2b(token.encode("utf-8"), digest_size=8).digest()
            value = int.from_bytes(digest, byteorder="little", signed=False)
            matrix[row, value % dimensions] += 1.0 if value & 1 else -1.0
    return normalize_rows(matrix)


def _mean_gensim_embeddings(frame: pd.DataFrame, model_name: str) -> np.ndarray:
    import gensim.downloader as downloader

    model = downloader.load(model_name)
    rows = []
    for document in serialize_cases(frame):
        vectors = []
        for token in re.findall(r"(?u)\b\w\w+\b", document.lower()):
            try:
                vectors.append(model.get_vector(token))
            except KeyError:
                continue
        rows.append(np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size))
    return normalize_rows(np.vstack(rows))


def _mean_flair_embeddings(frame: pd.DataFrame, vectoriser_name: str) -> np.ndarray:
    from flair.data import Sentence
    from flair.embeddings import FlairEmbeddings, StackedEmbeddings
    try:
        from flair.embeddings.legacy import ELMoEmbeddings
    except ImportError:
        from flair.embeddings import ELMoEmbeddings

    if vectoriser_name == "ELMo":
        embedding = ELMoEmbeddings()
    elif vectoriser_name == "Flair":
        embedding = StackedEmbeddings([
            FlairEmbeddings("news-forward"), FlairEmbeddings("news-backward"),
        ])
    else:
        raise ValueError(f"Unknown Flair vectoriser: {vectoriser_name}")

    rows = []
    for document in serialize_cases(frame):
        sentence = Sentence(document)
        embedding.embed(sentence)
        vectors = [token.embedding.detach().cpu().numpy() for token in sentence]
        rows.append(np.mean(vectors, axis=0) if vectors else np.zeros(embedding.embedding_length))
        sentence.clear_embeddings()
    return normalize_rows(np.vstack(rows))


representation_builders = {
    "TF-IDF": lambda: build_tfidf(cases),
    "Feature Hashing": lambda: build_feature_hashing(cases),
    "Word2Vec": lambda: _mean_gensim_embeddings(cases, "word2vec-google-news-300"),
    "GloVe": lambda: _mean_gensim_embeddings(cases, "glove-wiki-gigaword-300"),
    "FastText": lambda: _mean_gensim_embeddings(cases, "fasttext-wiki-news-subwords-300"),
    "ELMo": lambda: _mean_flair_embeddings(cases, "ELMo"),
    "Flair": lambda: _mean_flair_embeddings(cases, "Flair"),
}

representations = {}
for name in SELECTED_VECTORISERS:
    print(f"Building {name}. Pretrained methods may download their weights on this step.")
    representations[name] = representation_builders[name]()
    print(f"Finished {name}.")

for name, matrix in representations.items():
    assert matrix.shape[0] == len(cases), f"{name} row mismatch"
    assert np.isfinite(matrix).all(), f"{name} contains non-finite values"
    print(f"{name:20s} -> {matrix.shape}")

Building TF-IDF. Pretrained methods may download their weights on this step.
Finished TF-IDF.
Building Feature Hashing. Pretrained methods may download their weights on this step.
Finished Feature Hashing.
Building Word2Vec. Pretrained methods may download their weights on this step.
[==------------------------------------------------] 5.5% 91.0/1662.8MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[===-----------------------------------------------] 7.1% 117.7/1662.8MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[====----------------------------------------------] 8.9% 147.6/1662.8MB downloaded

...

[==================================================] 100.0% 1662.8/1662.8MB downloaded
Finished Word2Vec.
Building GloVe. Pretrained methods may download their weights on this step.
[==================================================] 100.0% 376.1/376.1MB downloaded
Finished GloVe.
Building FastText. Pretrained methods may download their weights on this step.
[==------------------------------------------------] 4.3% 41.2/958.4MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[===-----------------------------------------------] 8.0% 76.6/958.4MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[=================================-----------------] 66.7% 639.3/958.4MB downloaded

## 6. Controlled sequential selection scaffold

This pilot retains a Gaussian Process and UCB only as a controlled sequential scaffold for visualizing the effect of each representation. It is not the Bayesian Optimization stage of the research and must not be reported as a validated final classifier.

The run starts from the known warm-start outcomes. It then selects the next untested candidate with cost-aware UCB:

utility = clipped_mean + kappa × uncertainty

acquisition = utility / cost^cost_exponent

The notebook uses kappa = 1.5 and a square-root cost penalty. The predicted mean favors candidates that may expose a bug. Uncertainty lets the optimizer explore less familiar cases. Dividing by cost gives a mild preference to shorter tests.

In [ ]:
import numpy as np
import pandas as pd


def _squared_distances(left: np.ndarray, right: np.ndarray) -> np.ndarray:
    return np.maximum(
        np.sum(left * left, axis=1)[:, None]
        + np.sum(right * right, axis=1)[None, :]
        - 2.0 * left @ right.T,
        0.0,
    )


def _gp_predict(train_x, train_y, candidate_x, length_scale=1.0, noise=1e-3):
    """Fixed-kernel Gaussian Process posterior implemented with NumPy."""
    train_x = np.asarray(train_x, dtype=float)
    candidate_x = np.asarray(candidate_x, dtype=float)
    train_y = np.asarray(train_y, dtype=float)
    train_kernel = np.exp(-0.5 * _squared_distances(train_x, train_x) / length_scale**2)
    train_kernel += (noise + 1e-8) * np.eye(len(train_x))
    cross_kernel = np.exp(-0.5 * _squared_distances(train_x, candidate_x) / length_scale**2)
    cholesky = np.linalg.cholesky(train_kernel)
    weights = np.linalg.solve(cholesky.T, np.linalg.solve(cholesky, train_y))
    mean = cross_kernel.T @ weights
    projected = np.linalg.solve(cholesky, cross_kernel)
    variance = np.maximum(1.0 - np.sum(projected * projected, axis=0), 1e-12)
    return mean, np.sqrt(variance)


def _finalize_order(rows: list[dict]) -> pd.DataFrame:
    result = pd.DataFrame(rows)
    result["execution_position"] = np.arange(1, len(result) + 1)
    result["cumulative_bugs"] = result["confirmed_bug_dummy"].cumsum()
    result["cumulative_cost"] = result["estimated_cost"].cumsum()
    return result[
        [
            "execution_position", "tcs_id", "menu", "estimated_cost",
            "is_initial_seed", "predicted_mean", "predicted_std", "acquisition",
            "confirmed_bug_dummy", "cumulative_bugs", "cumulative_cost",
        ]
    ]


def run_closed_loop(
    matrix,
    ids,
    menus,
    costs,
    oracle,
    initial_indices,
    kappa=1.5,
    cost_exponent=0.5,
    snapshot_counts=None,
    random_state=42,
):
    """Return a full order while revealing only selected oracle labels."""
    matrix = np.asarray(matrix, dtype=float)
    ids = np.asarray(ids, dtype=str)
    menus = np.asarray(menus, dtype=str)
    costs = np.asarray(costs, dtype=float)
    oracle = np.asarray(oracle, dtype=int)
    if not (len(matrix) == len(ids) == len(menus) == len(costs) == len(oracle)):
        raise ValueError("matrix, ids, menus, costs, and oracle must have equal lengths")
    if np.any(costs <= 0):
        raise ValueError("All estimated costs must be positive")

    revealed = list(dict.fromkeys(int(index) for index in initial_indices))
    if not revealed:
        raise ValueError("At least one initial test is required")
    snapshot_counts = set(snapshot_counts or [])
    audit = []
    snapshots = []
    rows = [
        {
            "tcs_id": ids[index],
            "menu": menus[index],
            "estimated_cost": costs[index],
            "is_initial_seed": True,
            "predicted_mean": np.nan,
            "predicted_std": np.nan,
            "acquisition": np.nan,
            "confirmed_bug_dummy": int(oracle[index]),
        }
        for index in revealed
    ]

    while len(revealed) < len(ids):
        training_indices = list(revealed)
        mean, std = _gp_predict(matrix[training_indices], oracle[training_indices], matrix)

        remaining = np.array([index for index in range(len(ids)) if index not in set(revealed)], dtype=int)
        utility = np.clip(mean, 0.0, 1.0) + kappa * std
        acquisition = np.full(len(ids), np.nan, dtype=float)
        acquisition[remaining] = utility[remaining] / np.power(
            np.maximum(costs[remaining], 1.0), cost_exponent
        )
        best_value = np.nanmax(acquisition[remaining])
        tied = remaining[np.isclose(acquisition[remaining], best_value)]
        selected = int(min(tied, key=lambda index: ids[index]))

        audit.append(
            {
                "train_count": len(training_indices),
                "revealed_count": len(revealed),
                "training_indices": list(training_indices),
                "revealed_indices": list(revealed),
                "selected_index": selected,
            }
        )
        if len(revealed) in snapshot_counts:
            snapshots.append(
                {
                    "tested_count": len(revealed),
                    "observed_indices": list(revealed),
                    "selected_index": selected,
                    "mean": mean.copy(),
                    "std": std.copy(),
                    "acquisition": acquisition.copy(),
                }
            )

        rows.append(
            {
                "tcs_id": ids[selected],
                "menu": menus[selected],
                "estimated_cost": costs[selected],
                "is_initial_seed": False,
                "predicted_mean": float(mean[selected]),
                "predicted_std": float(std[selected]),
                "acquisition": float(acquisition[selected]),
                "confirmed_bug_dummy": int(oracle[selected]),
            }
        )
        revealed.append(selected)

    return _finalize_order(rows), audit, snapshots



## 7. How each vectoriser chooses test #10

All seven vectorisers receive the same nine warm-start outcomes. They differ only in how they judge similarity between an untested case and the known cases.

The table below shows each method's first candidate after warm start. Its final score is the cost-aware UCB score, so a candidate is preferred when it has a promising predicted result, useful uncertainty, and lower estimated test time.


In [ ]:
method_reasoning = pd.DataFrame([
    {"method": "TF-IDF", "what_it_compares": "Weighted shared words and word pairs"},
    {"method": "Feature Hashing", "what_it_compares": "Hashed word and word-pair counts"},
    {"method": "Word2Vec", "what_it_compares": "Average pretrained local-context word meaning"},
    {"method": "GloVe", "what_it_compares": "Average pretrained global co-occurrence word meaning"},
    {"method": "FastText", "what_it_compares": "Average pretrained subword-aware word meaning"},
    {"method": "ELMo", "what_it_compares": "Context-sensitive token meaning"},
    {"method": "Flair", "what_it_compares": "Context-sensitive character language-model meaning"},
])
display(method_reasoning)

first_choice_rows = []
remaining_after_warm_start = np.array([
    index for index in range(len(cases)) if index not in set(initial_indices)
])
for method_name, matrix in representations.items():
    mean, std = _gp_predict(matrix[initial_indices], oracle[initial_indices], matrix)
    utility = np.clip(mean, 0.0, 1.0) + KAPPA * std
    acquisition = utility / np.power(np.maximum(costs, 1.0), COST_EXPONENT)
    selected = int(remaining_after_warm_start[np.argmax(acquisition[remaining_after_warm_start])])
    first_choice_rows.append({
        "method": method_name,
        "test_10_candidate": ids[selected],
        "menu": menus[selected],
        "predicted_mean": float(np.clip(mean[selected], 0.0, 1.0)),
        "uncertainty": float(std[selected]),
        "estimated_cost_minutes": float(costs[selected]),
        "cost_aware_ucb_score": float(acquisition[selected]),
    })
first_choice_table = pd.DataFrame(first_choice_rows)
display(first_choice_table.style.format({
    "predicted_mean": "{:.3f}", "uncertainty": "{:.3f}",
    "estimated_cost_minutes": "{:.1f}", "cost_aware_ucb_score": "{:.3f}",
}))


## 8. Run the controlled sequential selection scaffold

Each method now continues from the same warm start. It selects one untested case, receives that case's dummy outcome, updates its Gaussian Process, and makes the next choice.


In [ ]:
snapshot_counts = {
    len(initial_indices),
    min(len(cases) - 1, len(initial_indices) + 5),
    min(len(cases) - 1, len(initial_indices) + 15),
    min(len(cases) - 1, len(initial_indices) + 30),
}

runs = {}
audits = {}
snapshots_by_method = {}
for method_name, matrix in representations.items():
    run, audit, snapshots = run_closed_loop(
        matrix=matrix,
        ids=ids,
        menus=menus,
        costs=costs,
        oracle=oracle,
        initial_indices=initial_indices,
        kappa=KAPPA,
        cost_exponent=COST_EXPONENT,
        snapshot_counts=snapshot_counts,
        random_state=RANDOM_STATE,
    )
    runs[method_name] = run
    audits[method_name] = audit
    snapshots_by_method[method_name] = snapshots


for method_name, run in runs.items():
    assert set(run["tcs_id"]) == set(cases["TCS ID"])
    assert run["tcs_id"].is_unique
    assert run["cumulative_bugs"].is_monotonic_increasing
    assert run["cumulative_cost"].is_monotonic_increasing
    assert int(run["cumulative_bugs"].iloc[-1]) == int(oracle.sum())
    print(f"{method_name:20s}: first dummy bug at test #{int(run.loc[run['confirmed_bug_dummy'].eq(1), 'execution_position'].min())}")


## 9. Convergence plot

This chart shows how many dummy bugs each method has found after each executed test. A curve that rises earlier has found more of the fixed dummy bugs with fewer test executions. Random selection provides a baseline.

In [ ]:
colors = dict(zip(PAPER_VECTORISERS, sns.color_palette("tab10", n_colors=len(PAPER_VECTORISERS))))

fig, ax = plt.subplots(figsize=(11, 6))
for method_name, run in runs.items():
    ax.step(run["execution_position"], run["cumulative_bugs"], where="post",
            linewidth=2.2, color=colors.get(method_name), label=method_name)

ax.axhline(oracle.sum(), color="crimson", linestyle="--", alpha=0.7, label="All dummy bugs")
ax.set(title="Convergence: Cumulative Dummy Bugs vs Executed Tests",
       xlabel="Number of executed test cases", ylabel="Cumulative dummy bugs found")
ax.legend()
fig.tight_layout()
fig.savefig(RESULT_DIR / "convergence.png", dpi=180, bbox_inches="tight")
plt.show()

## 10. Snapshot setup

The next figure uses the fixed Excel order to place the 69 discrete candidates on one readable axis. Connecting lines are visual guides. They do not turn the test cases into a continuous objective function.

In [ ]:
print("Snapshot counts:", sorted(snapshot_counts))

## 11. Model estimate and selection-score snapshots

Each representation gets its own figure. The left column shows the fixed dummy truth, the Gaussian Process mean, its uncertainty band, and the outcomes revealed so far. The right column shows the selection score used to choose the next test.

The figures below only compare the seven independent vectorisers.

In [ ]:
for method_name, method_snapshots in snapshots_by_method.items():
    fig, axes = plt.subplots(
        len(method_snapshots),
        2,
        figsize=(15, 3.8 * len(method_snapshots)),
        squeeze=False,
    )

    for row, snapshot in enumerate(method_snapshots):
        mean = snapshot["mean"]
        std = snapshot["std"]
        display_mean = np.clip(mean, 0.0, 1.0)
        display_lower = np.clip(mean - 1.96 * std, 0.0, 1.0)
        display_upper = np.clip(mean + 1.96 * std, 0.0, 1.0)
        observed = np.array(snapshot["observed_indices"], dtype=int)
        selected = snapshot["selected_index"]
        display_acquisition = np.nan_to_num(snapshot["acquisition"], nan=0.0)

        left = axes[row, 0]
        left.step(
            x_display,
            oracle,
            where="mid",
            color="red",
            linestyle="--",
            linewidth=1.5,
            label="True dummy value (unknown to BO)",
        )
        left.plot(x_display, display_mean, color="green", linestyle="--", linewidth=1.7, label="GP mean")
        left.fill_between(
            x_display,
            display_lower,
            display_upper,
            color="green",
            alpha=0.18,
            label="95% uncertainty band",
        )
        left.scatter(observed, oracle[observed], color="red", s=34, zorder=4, label="Observations")
        left.set_ylim(-0.05, 1.05)
        left.set_title(f"Posterior after {snapshot['tested_count']} observed tests")
        left.set_xlabel("Test case index in fixed Excel order")
        left.set_ylabel("Dummy outcome and GP score")
        if row == 0:
            left.legend(loc="upper right", fontsize=8)

        right = axes[row, 1]
        right.plot(x_display, display_acquisition, color="blue", linewidth=1.8, label="Acquisition E(x)")
        right.fill_between(x_display, 0, display_acquisition, color="blue", alpha=0.3)
        right.scatter(
            [selected],
            [display_acquisition[selected]],
            color="blue",
            s=70,
            zorder=4,
            label=f"Next query: {ids[selected]}",
        )
        right.set_title("Acquisition and next test case")
        right.set_xlabel("Test case index in fixed Excel order")
        right.set_ylabel("Cost-aware UCB acquisition")
        right.set_ylim(bottom=0)
        right.legend(loc="upper right", fontsize=8)

    fig.suptitle(f"{method_name}: Gaussian Process and acquisition", y=1.002, fontsize=15)
    fig.tight_layout()
    safe_name = re.sub(r"[^a-z0-9]+", "_", method_name.lower()).strip("_")
    fig.savefig(RESULT_DIR / f"iteration_snapshots_{safe_name}.png", dpi=180, bbox_inches="tight")
    plt.show()

## 12. Seven-vectoriser visual-demo summary

This is the final comparison table for the fixed semantic dummy demonstration. The Best Method marker ranks this one demonstration by the most dummy bugs found by run 20, then by run 50, the earliest run that finds all dummy bugs, and the lowest total cost. It is not a final representation decision.

In [ ]:
summary_rows = []
total_dummy_bug_count = int(oracle.sum())
for method_name, run in runs.items():
    bugs_by_20 = int(run.loc[run["execution_position"].le(20), "cumulative_bugs"].max())
    bugs_by_50 = int(run.loc[run["execution_position"].le(50), "cumulative_bugs"].max())
    all_found_row = run.loc[run["cumulative_bugs"].eq(total_dummy_bug_count)].iloc[0]
    summary_rows.append({
        "Method": method_name,
        "Dummy Bugs Found by Run 20": bugs_by_20,
        "Dummy Bugs Found by Run 50": bugs_by_50,
        "Run When All Dummy Bugs Were Found": int(all_found_row["execution_position"]),
        "Total Cost to Find All Dummy Bugs (minutes)": float(all_found_row["cumulative_cost"]),
    })

final_summary = pd.DataFrame(summary_rows).sort_values(
    by=[
        "Dummy Bugs Found by Run 20",
        "Dummy Bugs Found by Run 50",
        "Run When All Dummy Bugs Were Found",
        "Total Cost to Find All Dummy Bugs (minutes)",
    ],
    ascending=[False, False, True, True],
    kind="stable",
).reset_index(drop=True)
best_method = final_summary.iloc[0]["Method"]
final_summary.insert(1, "Best Method", np.where(final_summary["Method"].eq(best_method), "Yes", ""))

print(f"Best Method: {best_method}")
display(final_summary.style.format({
    "Total Cost to Find All Dummy Bugs (minutes)": "{:.1f}",
}))

for method_name, run in runs.items():
    safe_name = re.sub(r"[^a-z0-9]+", "_", method_name.lower()).strip("_")
    run.to_csv(RESULT_DIR / f"ordering_{safe_name}.csv", index=False)
final_summary.to_csv(RESULT_DIR / "final_summary.csv", index=False)

config = {
    "random_state": RANDOM_STATE,
    "kappa": KAPPA,
    "cost_exponent": COST_EXPONENT,
    **build_vectoriser_metadata(VECTORISER_SELECTION),
    "paper_vectorisers": list(PAPER_VECTORISERS),
    "paper_citation": PAPER_CITATION,
    "dummy_fault_themes": DUMMY_FAULT_THEMES,
    "dummy_bug_ids": DUMMY_BUG_IDS,
    "initial_tcs_ids": cases.iloc[initial_indices]["TCS ID"].tolist(),
    "warm_start_bug_count": warm_start_bug_count,
}
with (RESULT_DIR / "experiment_config.json").open("w", encoding="utf-8") as handle:
    json.dump(config, handle, indent=2, ensure_ascii=False)

current_artifacts = [
    RESULT_DIR / "convergence.png",
    RESULT_DIR / "final_summary.csv",
    RESULT_DIR / "experiment_config.json",
]
for method_name in runs:
    safe_name = re.sub(r"[^a-z0-9]+", "_", method_name.lower()).strip("_")
    current_artifacts.extend(
        [
            RESULT_DIR / f"ordering_{safe_name}.csv",
            RESULT_DIR / f"iteration_snapshots_{safe_name}.png",
        ]
    )
for artifact in current_artifacts:
    if not artifact.exists():
        raise FileNotFoundError(f"Expected current artifact was not created: {artifact}")

archive_path = RESULT_DIR.parent / "bayesian_dummy_results.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for artifact in current_artifacts:
        archive.write(artifact, arcname=artifact.name)
print("Results:", RESULT_DIR)
print("ZIP:", archive_path)
display(FileLink(str(archive_path)))

if AUTO_DOWNLOAD_IN_COLAB:
    try:
        from google.colab import files
        files.download(str(archive_path))
    except ImportError:
        pass

## 13. Non-visual synthetic fairness check

This section adds no plots. It is a neutral negative-control check for the seven vectorisers. **Neutral random fault instances** are created without reading the test-case text, representation matrices, model scores, or execution costs. Within each replicate, every method receives exactly the same fault labels and the same feature-covering warm start.

The check answers one narrow question: does the ranking from the fixed semantic dummy demonstration depend entirely on that one hand-placed fault pattern? It does not imitate real Android bugs. **No representation winner is declared** from these neutral synthetic labels; a final decision requires black-box replay outcomes from versioned buggy and fixed APK builds.


In [ ]:
import numpy as np


def draw_uniform_oracle(case_count, bug_count, random_state):
    """Draw neutral dummy labels without inspecting any representation."""
    if not 0 < bug_count <= case_count:
        raise ValueError("bug_count must be between 1 and case_count")
    rng = np.random.default_rng(random_state)
    oracle = np.zeros(case_count, dtype=int)
    oracle[rng.choice(case_count, size=bug_count, replace=False)] = 1
    return oracle


def choose_paired_initial_indices(menus, random_state):
    """Choose one shared warm-start case per feature/menu."""
    menus = np.asarray(menus, dtype=str)
    rng = np.random.default_rng(random_state)
    selected = []
    for menu in sorted(np.unique(menus)):
        candidates = np.flatnonzero(menus == menu)
        selected.append(int(rng.choice(candidates)))
    return selected


In [ ]:
N_PAIRED_FAIRNESS_REPLICATES = int(os.getenv("THREE_METHOD_FAIRNESS_REPLICATES", "10"))
FAIRNESS_BASE_SEED = 20260813
FAIRNESS_BUG_COUNT = 13

if N_PAIRED_FAIRNESS_REPLICATES < 2:
    raise ValueError("Use at least two paired fairness replicates.")

fairness_rows = []
for replicate in range(N_PAIRED_FAIRNESS_REPLICATES):
    instance_seed = FAIRNESS_BASE_SEED + replicate
    neutral_oracle = draw_uniform_oracle(
        case_count=len(cases),
        bug_count=FAIRNESS_BUG_COUNT,
        random_state=instance_seed,
    )
    paired_initial_indices = choose_paired_initial_indices(menus, instance_seed)

    for method_name, matrix in representations.items():
        fair_run, _, _ = run_closed_loop(
            matrix=matrix,
            ids=ids,
            menus=menus,
            costs=costs,
            oracle=neutral_oracle,
            initial_indices=paired_initial_indices,
            kappa=KAPPA,
            cost_exponent=COST_EXPONENT,
            snapshot_counts=set(),
            random_state=instance_seed,
        )
        all_found_row = fair_run.loc[
            fair_run["cumulative_bugs"].eq(FAIRNESS_BUG_COUNT)
        ].iloc[0]
        fairness_rows.append({
            "replicate": replicate + 1,
            "instance_seed": instance_seed,
            "Method": method_name,
            "Dummy Bugs Found by Run 20": int(
                fair_run.loc[fair_run["execution_position"].le(20), "cumulative_bugs"].max()
            ),
            "Dummy Bugs Found by Run 50": int(
                fair_run.loc[fair_run["execution_position"].le(50), "cumulative_bugs"].max()
            ),
            "Run When All 13 Dummy Bugs Were Found": int(
                all_found_row["execution_position"]
            ),
            "Total Cost to Find All 13 Dummy Bugs (minutes)": float(
                all_found_row["cumulative_cost"]
            ),
        })

fairness_runs = pd.DataFrame(fairness_rows)
fairness_summary = fairness_runs.groupby("Method", as_index=False).agg(
    **{
        "Mean Bugs Found by Run 20": ("Dummy Bugs Found by Run 20", "mean"),
        "Mean Bugs Found by Run 50": ("Dummy Bugs Found by Run 50", "mean"),
        "Median Run When All 13 Dummy Bugs Were Found": (
            "Run When All 13 Dummy Bugs Were Found", "median"
        ),
        "Mean Total Cost to Find All 13 Dummy Bugs (minutes)": (
            "Total Cost to Find All 13 Dummy Bugs (minutes)", "mean"
        ),
    }
).sort_values("Method", kind="stable").reset_index(drop=True)

fairness_runs.to_csv(RESULT_DIR / "synthetic_fairness_runs.csv", index=False)
fairness_summary.to_csv(RESULT_DIR / "synthetic_fairness_summary.csv", index=False)

config["paired_fairness_replicates"] = N_PAIRED_FAIRNESS_REPLICATES
config["paired_fairness_base_seed"] = FAIRNESS_BASE_SEED
config["paired_fairness_oracle"] = "uniform_random_without_representation_features"
with (RESULT_DIR / "experiment_config.json").open("w", encoding="utf-8") as handle:
    json.dump(config, handle, indent=2, ensure_ascii=False)

current_artifacts.extend([
    RESULT_DIR / "synthetic_fairness_runs.csv",
    RESULT_DIR / "synthetic_fairness_summary.csv",
])
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for artifact in current_artifacts:
        archive.write(artifact, arcname=artifact.name)

print("No representation winner is declared from neutral synthetic labels.")
display(fairness_summary.style.format({
    "Mean Bugs Found by Run 20": "{:.2f}",
    "Mean Bugs Found by Run 50": "{:.2f}",
    "Median Run When All 13 Dummy Bugs Were Found": "{:.1f}",
    "Mean Total Cost to Find All 13 Dummy Bugs (minutes)": "{:.1f}",
}))


## 14. How to read the report

Read the visual-demo table and the neutral synthetic table separately. The first explains how each representation behaves on the deliberately semantic dummy scenario. The second checks whether that one scenario alone drives the apparent ranking. Neither table proves which representation will find real Android bugs earlier.
